# Sprint Block Q — Multi-source dataset

Builds `corpus_v3`: Wikidata (strong P31 labels) + Wikipedia leads + the existing Wikivoyage
weak-labelled rows + the PK slice. Then Q5 re-runs the whole pipeline once, cleanly.

| Q | source | label kind | runtime |
|---|---|---|---|
| **Q1** | Wikidata SPARQL — `description` + `P31 instance-of` + coords | **strong** | CPU |
| Q2 | Wikipedia geo-article leads (reuse Block B1 parser) | medium (infobox) | CPU |
| Q3 | Reddit — separate collection agent | — | — |
| Q4 | merge → `corpus_v3` (`text, source, region, label, label_source, lat, lon`) | | CPU/GPU |
| Q5 | consolidated pipeline: FP-Growth → BEN → ranker → eval on the 106 | | GPU |
| Q6 | v1 (Wikivoyage) vs v3 (multi-source) on the 106 | | CPU |

---

## Q1 — Wikidata harvest

Wikidata items that: are `instance of` a place type we care about (`?item wdt:P31 ?type`), have a
**short description** (`schema:description` in English), and have coordinates (`wdt:P625`).
The description is paraphrase-like (*"Mughal-era fort in Lahore"*); `P31` gives a clean type;
the coordinate lets us tag `region`.

First: the P31→OSM-tag lookup, then a LIMIT-50 probe to confirm the query shape.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, re, time
import numpy as np, pandas as pd
from collections import Counter
ROOT = '/content/drive/MyDrive/geoproject'
os.makedirs(f'{ROOT}/raw_q', exist_ok=True)          # Phase Q raw harvests
os.makedirs(f'{ROOT}/results_q', exist_ok=True)
print('ready:', ROOT)


In [ ]:
# P31 (Wikidata "instance of") QID -> OSM tag  (aligned to the 227-tag eval lexicon)
LEX = set(json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon'])

P31_TO_TAG = {
    'Q11707':'restaurant', 'Q30022':'cafe', 'Q212198':'fast-food', 'Q207694':'museum',
    'Q41253':'theatre', 'Q17350442':'theatre', 'Q1195942':'cinema', 'Q7075':'library',
    'Q7857':'shrine', 'Q44613':'monastery', 'Q160742':'abbey', 'Q2977':'cathedral',
    'Q16970':'church', 'Q32815':'mosque', 'Q34627':'shrine', 'Q842402':'temple',
    'Q44539':'temple', 'Q1088552':'gurdwara', 'Q120560':'chapel', 'Q108325':'chapel',
    'Q23413':'castle', 'Q57821':'fort', 'Q1785071':'fort', 'Q744099':'palace', 'Q16560':'palace',
    'Q57831':'citadel', 'Q4989906':'monument', 'Q5003624':'monument', 'Q381885':'tomb',
    'Q381681':'ruins', 'Q839954':'attraction', 'Q570116':'attraction', 'Q2319498':'attraction',
    'Q33506':'museum', 'Q207694':'museum', 'Q1007870':'gallery', 'Q1329623':'arts-center',
    'Q22698':'park', 'Q22746':'garden', 'Q1783497':'garden', 'Q3010369':'garden',
    'Q40080':'beach', 'Q4421':'forest', 'Q23397':'lake', 'Q23442':'lake', 'Q4022':'river',
    'Q12284':'canal', 'Q8502':'mountain', 'Q54050':'hill', 'Q39816':'valley', 'Q179700':'statue',
    'Q40357':'prison', 'Q16917':'hospital', 'Q3914':'school', 'Q3918':'university', 'Q9842':'school',
    'Q7930989':'city', 'Q515':'city', 'Q3957':'town', 'Q532':'village', 'Q5084':'hamlet',
    'Q41176':'building', 'Q11303':'skyscraper', 'Q11315':'mall', 'Q7075':'library',
    'Q11315':'mall', 'Q205495':'gas-station', 'Q11readonly':'',   # placeholder guard
    'Q483110':'stadium', 'Q641226':'arena', 'Q1076486':'stadium',
    'Q40970':'amusement-park', 'Q194195':'amusement-park', 'Q43229':'organization',
    'Q11315':'mall', 'Q7278':'political-party',
    'Q27686':'hotel', 'Q1543677':'guest-house', 'Q1889035':'hostel',
    'Q37937':'lighthouse', 'Q12518':'tower', 'Q1440300':'observation-tower',
    'Q174782':'square', 'Q79007':'street', 'Q83620':'road', 'Q34442':'road', 'Q12280':'bridge',
    'Q55488':'station', 'Q928830':'metro-station', 'Q55485':'railway-station', 'Q17158460':'stop',
    'Q1358919':'brewery', 'Q131734':'brewery', 'Q857909':'winery', 'Q373899':'pub', 'Q212198':'fast-food',
    'Q187456':'bar', 'Q35476':'observatory', 'Q1244442':'school', 'Q159719':'power-station',
    'Q483453':'marketplace', 'Q330284':'marketplace', 'Q132510':'bazaar', 'Q8514':'desert',
    'Q39614':'cemetery', 'Q1076486':'stadium', 'Q47521':'stream', 'Q185113':'cape',
    'Q4830453':'business', 'Q43501':'zoo', 'Q1972120':'aquarium', 'Q207326':'botanical-garden',
}
# keep only mappings whose target is an actual eval-lexicon tag
P31_TO_TAG = {k: v for k, v in P31_TO_TAG.items() if v in LEX}
print(f'P31->tag entries landing in the 227-tag lexicon: {len(P31_TO_TAG)}')
print('distinct target tags:', sorted(set(P31_TO_TAG.values())))
json.dump(P31_TO_TAG, open(f'{ROOT}/results_q/p31_to_tag.json', 'w'), indent=1)


In [ ]:
import requests

# WDQS is in an active outage (1 req/min). QLever (U. Freiburg) is an independent
# fast Wikidata SPARQL endpoint.
QLEVER = 'https://qlever.cs.uni-freiburg.de/api/wikidata'
PREFIXES = ("PREFIX wd: <http://www.wikidata.org/entity/> "
            "PREFIX wdt: <http://www.wikidata.org/prop/direct/> "
            "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> "
            "PREFIX schema: <http://schema.org/> ")
_S = requests.Session()
_S.headers.update({'User-Agent': 'geoproject-thesis/1.0 (research)'})

def sparql(q, timeout=180):
    r = _S.post(QLEVER, data={'query': PREFIXES + q},
                headers={'Accept': 'application/sparql-results+json'}, timeout=timeout)
    r.raise_for_status()
    return r.json()['results']['bindings']

probe_q = """
SELECT ?item ?label ?desc ?coord WHERE {
  ?item wdt:P31 wd:Q57831 ;
        wdt:P625 ?coord ;
        schema:description ?desc ;
        rdfs:label ?label .
  FILTER(LANG(?desc) = "en")
  FILTER(LANG(?label) = "en")
} LIMIT 50
"""
t0 = time.time(); rows = sparql(probe_q)
print(f'{len(rows)} rows in {time.time()-t0:.1f}s\n')
for b in rows[:8]:
    print(f"  {b['label']['value']:<28} | {b['desc']['value'][:58]:<58} | {b['coord']['value']}")


## Q1.2 — Batched harvest, one query per place type

For each P31 type: items with an English `schema:description` + `wdt:P625` coordinates, `LIMIT
8000` (mega-types like church/city aren't exhausted — we want a balanced set, not every church on
Earth). ~1s pause between queries (WDQS etiquette). Coordinates → `region` (PK bbox + a light
country-name check). Saved to `raw_q/wikidata_places.parquet`.


In [ ]:
import time as _t

Q_TMPL = """
SELECT ?item ?label ?desc ?coord WHERE {{
  ?item wdt:P31 wd:{qid} ;
        wdt:P625 ?coord ;
        schema:description ?desc ;
        rdfs:label ?label .
  FILTER(LANG(?desc) = "en")
  FILTER(LANG(?label) = "en")
}} LIMIT 12000
"""

def parse_point(s):
    m = re.match(r'POINT\(([-\d.]+) ([-\d.]+)\)', s, re.I)
    return (float(m.group(2)), float(m.group(1))) if m else (None, None)   # (lat, lon)

# reload the PK boundary from Block M's saved geojson
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep
pk_shape = prep(gpd.read_file(f'{ROOT}/results_m/_pakistan.geojson').geometry.iloc[0].buffer(0.05))

rows, fails = [], []
targets = sorted(set((q, t) for q, t in P31_TO_TAG.items()), key=lambda x: x[1])
for i, (qid, tag) in enumerate(targets):
    for attempt in range(3):
        try:
            res = sparql(Q_TMPL.format(qid=qid), timeout=240)
            for b in res:
                lat, lon = parse_point(b['coord']['value'])
                rows.append({'qid': b['item']['value'].rsplit('/', 1)[-1],
                             'label': b['label']['value'], 'desc': b['desc']['value'],
                             'p31': qid, 'tag': tag, 'lat': lat, 'lon': lon})
            break
        except Exception as e:
            if attempt == 2:
                fails.append((qid, tag, str(e)[:100]))
            _t.sleep(3 * (attempt + 1))
    if i % 12 == 0:
        print(f'  [{i+1}/{len(targets)}] {tag:<14} total {len(rows):,}', flush=True)
    _t.sleep(0.5)

wd = pd.DataFrame(rows).drop_duplicates('qid').reset_index(drop=True)
wd = wd[wd.lat.notna()]
wd['region'] = np.where([pk_shape.contains(Point(x, y)) for x, y in zip(wd.lon, wd.lat)], 'pk', 'global')
print(f'\nWikidata places: {len(wd):,}  ({wd.tag.nunique()} tags, {(wd.region=="pk").sum():,} PK)')
if fails: print('failed:', fails)
print('\nrows per tag:'); print(wd.tag.value_counts().to_string())

wd.to_parquet(f'{ROOT}/raw_q/wikidata_places.parquet', index=False)
print(f'\nwrote raw_q/wikidata_places.parquet')


In [ ]:
wd = pd.read_parquet(f'{ROOT}/raw_q/wikidata_places.parquet')

TYPE_NOUNS = ('castle|fort|fortress|palace|temple|mosque|church|cathedral|shrine|tomb|monument|'
    'building|house|tower|bridge|market|bazaar|garden|park|museum|gallery|theatre|'
    'restaurant|cafe|hotel|pub|bar|library|school|university|hospital|station|street|'
    'road|square|village|town|city|river|lake|canal|hill|valley|forest|beach|stadium|'
    'quarter|district|hall|centre|center')
ADJ_PAT = re.compile(r'\b([a-z]+(?:-[a-z]+)?)\s+(?:' + TYPE_NOUNS + r')\b', re.I)
STOPADJ = {'the','a','an','of','in','on','at','and','or','to','is','was','by','for','its',
           'type','kind','sort','south','north','east','west','central'}

hits = wd.desc.str.lower().str.extract(ADJ_PAT.pattern)[0]
has_adj = hits.notna() & ~hits.isin(STOPADJ)
print(f'descriptions with an <adjective> <type-noun> pattern: {has_adj.sum():,} / {len(wd):,} '
      f'({has_adj.mean():.0%})')

adj_counts = Counter(a for a in hits[has_adj] if a not in STOPADJ)
print('\ntop adjectives before a type noun:')
for a, c in adj_counts.most_common(30):
    print(f'  {c:>5}  {a}')

print('\nsample adjective-bearing descriptions:')
for d in wd.desc[has_adj].sample(12, random_state=3):
    print(f'  {d[:95]}')


## Q3 + Q4 — extract adjectives from the Wikidata descriptions, merge → `corpus_v3`

Run the SAME `adjective_extractor.PlaceInterpreter.pipe()` used in Block D/H4 on the 497k Wikidata
`desc` strings. Then build `corpus_v3` = Wikidata rows (strong `P31` label) ∪ Wikivoyage rows
(weak `weak_label()` label, from `annotated_v2.parquet`), with a `label_source` column so every
downstream table can slice strong vs weak.


In [ ]:
%%capture
import torch
if torch.cuda.is_available():
    !pip install -q "spacy>=3.7,<3.9" spacy-transformers
    !python -m spacy download en_core_web_trf -q


In [ ]:
import sys, spacy, torch, json
from tqdm.auto import tqdm
sys.path.insert(0, ROOT)
import adjective_extractor as ax

if torch.cuda.is_available():
    spacy.require_gpu()
MODEL = 'en_core_web_trf'
pi = ax.PlaceInterpreter(model=MODEL)
print('extractor loaded, GPU:', torch.cuda.is_available())

wd = pd.read_parquet(f'{ROOT}/raw_q/wikidata_places.parquet')
# a Wikidata "sentence" = "<Label> is <description>." so the extractor sees a real NP subject
texts = [f"{r.label} is {r.desc}." if not r.desc[:1].isupper()
         else f"{r.label}: {r.desc}." for r in wd.itertuples(index=False)]

recs = []
t0 = time.time()
for i, it in enumerate(tqdm(pi.pipe(texts, batch_size=64), total=len(texts), desc='wikidata')):
    adjs = sorted({a for e in it.entities for a in e['adjectives']})
    recs.append({'adjectives': adjs, 'pairs': json.dumps(it.pairs),
                 'tag_hints': json.dumps(it.tag_hints),
                 'transaction': json.dumps(it.transaction)})
    if i and i % 50000 == 0:
        pd.DataFrame(recs).to_parquet(f'{ROOT}/raw_q/_wd_extract_ckpt.parquet')  # checkpoint
print(f'extracted {len(recs):,} in {(time.time()-t0)/60:.1f} min')

wd_x = pd.concat([wd.reset_index(drop=True), pd.DataFrame(recs)], axis=1)
n_adj = wd_x.adjectives.str.len()
print(f'mean adjectives/desc: {n_adj.mean():.2f}   with >=1: {(n_adj>0).mean():.0%}   '
      f'tag_hints on: {(wd_x.tag_hints != "[]").mean():.0%}')
wd_x.to_parquet(f'{ROOT}/raw_q/wikidata_extracted.parquet', index=False)
print('wrote raw_q/wikidata_extracted.parquet')


In [ ]:
# ============ Q4 — merge → corpus_v3 ======================================
wd_x = pd.read_parquet(f'{ROOT}/raw_q/wikidata_extracted.parquet')
wl   = pd.read_parquet(f'{ROOT}/results_m/weak_labelled_v2.parquet')   # Wikivoyage weak labels + split + region

rng = np.random.default_rng(0)

# --- Wikidata part (strong P31 labels) ---
wd_part = pd.DataFrame({
    'text':  [f"{r.label} is {r.desc}." for r in wd_x.itertuples(index=False)],
    'label': wd_x.tag,
    'label_source': 'wikidata_p31',
    'source': 'wikidata', 'region': wd_x.region,
    'pairs': wd_x.pairs, 'transaction': wd_x.transaction, 'tag_hints': wd_x.tag_hints,
})
# stratified 85/15 split by tag
wd_part['split'] = 'train'
dev_ix = (wd_part.groupby('label', group_keys=False)
          .apply(lambda g: g.sample(frac=0.15, random_state=0)).index)
wd_part.loc[dev_ix, 'split'] = 'dev'

# --- Wikivoyage part (weak lexicon labels) ---
wv_part = pd.DataFrame({
    'text': wl.text, 'label': wl.weak_tag, 'label_source': 'wikivoyage_lexicon',
    'source': 'wikivoyage', 'region': wl.region, 'split': wl.split,
    'pairs': wl.pairs, 'transaction': wl.transaction,
    'tag_hints': wl.get('tag_hints', pd.Series(['[]']*len(wl))),
})

corpus_v3 = pd.concat([wd_part, wv_part], ignore_index=True)
corpus_v3 = corpus_v3[corpus_v3.label.notna() & (corpus_v3.label != '')].reset_index(drop=True)
corpus_v3.to_parquet(f'{ROOT}/annotated/corpus_v3.parquet', index=False)

print(f'corpus_v3: {len(corpus_v3):,} rows')
print(corpus_v3.groupby(['label_source', 'split']).size().to_string())
print(f'\nregion: {corpus_v3.region.value_counts().to_dict()}')
print(f'distinct labels: {corpus_v3.label.nunique()}')
print(f'\nlabel distribution (top 20):')
print(corpus_v3.label.value_counts().head(20).to_string())
print(f'\nwrote annotated/corpus_v3.parquet')


In [ ]:
import os, pandas as pd
try:
    print('wikidata_extracted exists:', os.path.exists(f'{ROOT}/raw_q/wikidata_extracted.parquet'))
    if os.path.exists(f'{ROOT}/raw_q/_wd_extract_ckpt.parquet'):
        print('checkpoint rows:', len(pd.read_parquet(f'{ROOT}/raw_q/_wd_extract_ckpt.parquet')))
    print('recs in memory:', len(recs) if 'recs' in dir() else 'n/a')
except Exception as e:
    print(e)


In [ ]:
wd_x = pd.read_parquet(f'{ROOT}/raw_q/wikidata_extracted.parquet')
import numpy as np, json
n_adj = wd_x.adjectives.str.len()
print(f'wikidata_extracted: {len(wd_x):,} rows')
print(f'mean adjectives/desc: {n_adj.mean():.2f}   with >=1 adjective: {(n_adj>0).mean():.0%}')
print(f'tag_hints on: {(wd_x.tag_hints != "[]").mean():.0%}')
from collections import Counter
ac = Counter(a for row in wd_x.adjectives for a in row)
print('\ntop 25 extracted adjectives:')
for a,c in ac.most_common(25): print(f'  {c:>6}  {a}')


## Q5 — Consolidated pipeline on `corpus_v3`

Guided FP-Growth → EN feature selection → W2V + EN blend ranker → eval on the 106. Same code as
Block P, pointed at `corpus_v3`. Frequentist EN first (fast) for the immediate v1-vs-v3 answer;
Bayesian EN on the (cleaner, sharper) v3 pattern set after.


In [ ]:
import json, re, numpy as np, pandas as pd, time
from collections import Counter
from scipy import sparse

cv3 = pd.read_parquet(f'{ROOT}/annotated/corpus_v3.parquet')
tr3 = cv3[cv3.split == 'train'].reset_index(drop=True)
print(f'corpus_v3 train: {len(tr3):,}')

def adjs_of(pj):
    p = json.loads(pj) if isinstance(pj, str) else pj
    return sorted({a.lower() for a, n, rel in p if rel == 'amod' and str(a).replace('-','').isalpha()})
def amod_items(tj):
    t = json.loads(tj) if isinstance(tj, str) else tj
    return [it for it in t if it.startswith('amod:')]

tr3 = tr3.assign(adjs=tr3.pairs.apply(adjs_of), pit=tr3.transaction.apply(amod_items))
N = len(tr3); MIN_SUP = 0.0004; MIN_CNT = MIN_SUP * N
c1 = Counter(a for r in tr3.adjs for a in r)
c2 = Counter(p for r in tr3.pit for p in r)
def _ok(p):
    b = p.split(':',1)[1] if ':' in p else p
    return all(len(t) > 1 and t.isalnum() for t in b.replace('amod:','').split('_'))
FREQ_ADJ  = sorted(a for a,c in c1.items() if c >= MIN_CNT and _ok(a))
FREQ_PAIR = sorted(p for p,c in c2.items() if c >= MIN_CNT and _ok(p))
PATVOCAB = FREQ_ADJ + FREQ_PAIR
PIX = {p: j for j, p in enumerate(PATVOCAB)}
print(f'v3 guided FP-Growth patterns: {len(PATVOCAB)}  ({len(FREQ_ADJ)} adj + {len(FREQ_PAIR)} pair)  '
      f'[v1 had 966]')

ADJ_SET, PAIR_SET = set(FREQ_ADJ), set(FREQ_PAIR)
def featurize(df):
    ri, ci = [], []
    for r, row in enumerate(df.itertuples(index=False)):
        seen = {PIX[a] for a in adjs_of(row.pairs) if a in ADJ_SET}
        seen |= {PIX[p] for p in amod_items(row.transaction) if p in PAIR_SET}
        ri += [r]*len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(df), len(PATVOCAB)))

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(tr3.label)
Xtr = featurize(tr3); ytr = le.transform(tr3.label.values)
print(f'X_train {Xtr.shape}  coverage {(Xtr.getnnz(1)>0).mean():.0%}  classes {len(le.classes_)}')

ev = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json'))
def feat_eval(rows):
    ri, ci = [], []
    for r, x in enumerate(rows):
        seen = {PIX[a] for a in x.get('adjectives', []) if a in ADJ_SET}
        seen |= {PIX[p] for p in x.get('transaction', []) if p in PAIR_SET}
        ri += [r]*len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(rows), len(PATVOCAB)))
Xev = feat_eval(ev)
print(f'X_eval {Xev.shape}  coverage {(Xev.getnnz(1)>0).mean():.0%}')

sparse.save_npz(f'{ROOT}/results_q/Xtr_v3.npz', Xtr); sparse.save_npz(f'{ROOT}/results_q/Xev_v3.npz', Xev)
np.savez(f'{ROOT}/results_q/q5_meta.npz', patvocab=np.array(PATVOCAB), classes=le.classes_,
         n_adj=len(FREQ_ADJ), ytr=ytr)
print('saved results_q/Xtr_v3.npz, Xev_v3.npz, q5_meta.npz')


In [ ]:
import pickle
from itertools import groupby
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

# ---- frequentist EN on v3 patterns ----
rng = np.random.default_rng(42)
keep = (tr3.groupby('label', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), max(1, round(len(g)*150_000/len(tr3)))), random_state=42))
        .index.values)
t0 = time.time()
fen = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5,
                         max_iter=200, tol=1e-3, n_jobs=-1)
fen.fit(Xtr[keep], ytr[keep])
FEN = fen.coef_.astype(np.float32)
print(f'v3 frequentist EN: {Xtr[keep].shape}, {time.time()-t0:.0f}s, '
      f'{(FEN!=0).mean():.0%} nonzero, saga {int(np.max(fen.n_iter_))} iters')
sig = np.abs(FEN).max(0)
print('top patterns:', [PATVOCAB[j] for j in np.argsort(-sig)[:12]])
np.savez(f'{ROOT}/results_q/q5_frequentist_en.npz', coef=FEN, classes=le.classes_, patvocab=np.array(PATVOCAB))

# ---- W2V ranker (compact) ----
W2V = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset.pkl', 'rb'))
d106 = json.load(open(f'{ROOT}/eval/evalset_106.json')); LEX = d106['lexicon']
ABSTRACT = {'place','spot','location','area','site','position','thing','kind','sort','lot','bit','way','day','time'}
KS = (1,2,3,5,10,20)
def _pw(s):
    for v in [s,s.lower(),s.title(),s.capitalize(),s.replace(' ','_'),s.replace('-','_'),s.replace(' ','').replace('-','')]:
        if v in W2V: return W2V[v]
    sub = [W2V[w] for w in re.split(r'[\s\-_/]+', s) for w in [w,w.lower()] if w in W2V][:8]
    return np.mean(sub,0) if sub else None
WT = [t for t in LEX if _pw(t) is not None]
WM = np.vstack([_pw(t) for t in WT]); WM /= np.linalg.norm(WM,axis=1,keepdims=True)+1e-9
WIX = {t:i for i,t in enumerate(WT)}
def w2v_score(terms):
    s = np.zeros(len(WT), np.float32)
    for cn in terms:
        v = _pw(cn)
        if v is None: continue
        sim = WM @ (v/(np.linalg.norm(v)+1e-9)); s += (sim-sim.min())/(sim.max()-sim.min()+1e-9)
    return s
def metrics(r, ks=KS):
    r = np.asarray(r, float)
    o = {f'Hits@{k}': float((r<=k).mean()) for k in ks}
    o['nDCG@20'] = float(np.where(r<=20, 1/np.log2(r+1), 0).mean()); return o
CLS = list(le.classes_); C2W = {c: WIX[c] for c in CLS if c in WIX}
def en_score(coef, xrow):
    s = np.zeros(len(WT), np.float32); pj = xrow.nonzero()[1]
    if len(pj)==0: return s
    agg = coef[:, pj].sum(1)
    for c,w in C2W.items(): s[w] = agg[CLS.index(c)]
    lo,hi = s.min(), s.max()
    return (s-lo)/(hi-lo+1e-9) if hi>lo else s
def eval_blend(coef, lam):
    rk=[]
    for i,x in enumerate(ev):
        terms=[t for t in x['context_nouns'] if t.lower() not in ABSTRACT]
        s=w2v_score(terms); s=(s-s.min())/(s.max()-s.min()+1e-9)
        if lam>0: s=(1-lam)*s + lam*en_score(coef, Xev[i])
        g=x['gold_tag']; rk.append(int((s>=s[WIX[g]]).sum()) if g in WIX else len(WT)+1)
    return np.array(rk)
def cv_eval(coef, name):
    R={L:eval_blend(coef,L) for L in [0,.1,.2,.25,.3,.35,.4,.5]}
    kf=KFold(5,shuffle=True,random_state=42); idx=np.arange(len(ev)); h=np.zeros(len(ev)); ch=[]
    for tri,tei in kf.split(idx):
        L=max(R,key=lambda l:(R[l][tri]<=3).mean()); ch.append(L); h[tei]=R[L][tei]
    m=metrics(h.tolist())
    print(f'  {name:<22} λ~{np.median(ch):.2f}  H@1 {m["Hits@1"]:.3f}  H@3 {m["Hits@3"]:.3f}  '
          f'H@5 {m["Hits@5"]:.3f}  nDCG {m["nDCG@20"]:.3f}')
    return m

_d = Counter(x['gold_tag'] for x in ev)
print(f'\n{"Q5/Q6 — v3 pipeline vs v1":-^60}')
print(f'  {"baseline: freq prior":<22} H@1 {_d.most_common(1)[0][1]/106:.3f}  '
      f'H@3 {sum(sorted(_d.values(),reverse=True)[:3])/106:.3f}')
print(f'  {"v1 (Wikivoyage-only)":<22} H@1 0.311  H@3 0.509  H@5 0.613  nDCG 0.529')
r_base = cv_eval(np.zeros_like(FEN), 'v3 nouns-only')
r_v3   = cv_eval(FEN, 'v3 + freq-EN blend')
json.dump({'v3_baseline': r_base, 'v3_frequentist_blend': r_v3,
           'v1_reference': {'Hits@1':0.311,'Hits@3':0.509,'Hits@5':0.613,'nDCG@20':0.529}},
          open(f'{ROOT}/results_q/q6_v1_vs_v3.json','w'), indent=2)
print('\nwrote results_q/q6_v1_vs_v3.json')
